# Estudo de ablação — tarefas binárias (4 grupos)

Hipocampo L/R × T1/T2/T3 (features absolutas, sem deltas).

| Task | Classes |
|------|---------|
| `cn_ad` | CN (0) vs AD (1) |
| `smci_pmci` | sMCI (0) vs pMCI (1) |
| `cn_smci` | CN (0) vs sMCI (1) |
| `smci_ad` | sMCI (0) vs AD (1) |
| `cn_pmci` | CN (0) vs pMCI (1) |
| `pmci_ad` | pMCI (0) vs AD (1) |

Presets em `TASK_PRESETS`: `core` (2 within), `cross` (4 cross), `all` (6).

**Modalidades** (`csvs/longitudinal_4_groups/ablation/{roi}/`):

| ID | CSV long | Conteúdo |
|----|----------|----------|
| `vol` | `vol_long.csv` | volumétrico (gm/wm/csf) |
| `shape` | `shape_long.csv` | morfologia radiômica (`original_shape_*`) |
| `texture` | `rad_long.csv` | textura (GLCM, GLRLM, …) |
| `disp` | `disp_long.csv` | deslocamento |
| `all` | `merge_long.csv` | vol + shape + texture + disp |

**Fluxo:** CSV long → split `ID_PT` → ComBat opcional → pivot wide → nested CV 5×5 (× `R_REPEATS` repetições).

**Resultados:** `csvs/longitudinal_4_groups/ablation_results/{MODALITY}/`

Lógica em `ablation_runner.py` + análise em `ablation_analysis.py`.

In [ ]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning

from ablation_analysis import (
    METRIC_COLS,
    feature_freq_table,
    filter_ablation_config,
    plot_feature_stability,
    prepare_ablation_df,
    summary_with_pooled,
)
from ablation_prep import ROI_FILTER_DEFAULT
from ablation_runner import MODALITIES, SELECTION_MODES, TASKS, TASK_PRESETS, run_full_ablation_suite

warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
ROI = ROI_FILTER_DEFAULT
BASE_DIR = Path(f"csvs/longitudinal_4_groups/ablation/{ROI}")

# Alternar modalidade por run — resultados em subpasta separada
MODALITY = "vol"  # vol | shape | texture | disp | all
RUN_MODALITIES = (MODALITY,)
RESULTS_DIR = Path(f"csvs/longitudinal_4_groups/ablation_results/{MODALITY}")
RESULTS_PATH = RESULTS_DIR / "ablation_results_all.csv"
SUMMARY_PATH = RESULTS_DIR / "ablation_summary.csv"

# R=0 → nested 5×5 uma vez; R=10 → 10 repetições (50 avaliações externas por config)
R_REPEATS = 10

# Saída no console: False = silencioso (padrão)
VERBOSE = True
COMBAT_QUIET = True

# ComBat por fold — (False,) só sem; (False, True) compara ambos
WITH_COMBAT_FLAGS = (False, True)

# TASK_PRESETS: "core" | "cross" | "all" — ou tupla explícita de task_id
RUN_TASKS = TASK_PRESETS["core"]
RUN_SELECTION_MODES = ("mrmr",)  # "raw", "filters", "mrmr"
MODELS = ("svm", "rf", "mlp")  # "svm", "rf", "xgb", "mlp"

## Reanalisar sem retreinar

CSVs em `RESULTS_DIR` (ex.: `ablation_results/vol/`):

- `ablation_results_all.csv` → 1 linha por (repeat_id, fold)
- `ablation_summary.csv` → AUC média ± std **e AUC pooled** por configuração

**Fluxo rápido:** ajuste `MODALITY` → carregar CSV → gráficos (células 5–9).


In [ ]:
# # Carregar resultados salvos (pule a célula de execução)
# RESULTS_DIR.mkdir(parents=True, exist_ok=True)
# df_ablation = prepare_ablation_df(pd.read_csv(RESULTS_PATH))
# summary = summary_with_pooled(df_ablation)
# print(f"Carregado: {RESULTS_PATH} ({len(df_ablation)} linhas, {summary['n_repeats'].max()} repetições)")
# summary[["task", "model_key", "with_combat", "auc_mean", "auc_pooled"]].head()

In [ ]:
# Visão geral dos CSVs long de entrada (por task)
for mod_id in RUN_MODALITIES:
    cfg = MODALITIES[mod_id]
    long_path = BASE_DIR / cfg["long"]
    if not long_path.is_file():
        print(f"[{mod_id}] AUSENTE: {long_path}")
        continue
    df = pd.read_csv(long_path)
    print(f"\n{mod_id.upper():8s} — {cfg['label']} — {long_path.name}")
    print(f"  linhas={len(df):,} | imagens={df['ID_IMG'].nunique():,} | pacientes={df['ID_PT'].nunique():,}")
    for task_id in RUN_TASKS:
        task = TASKS[task_id]
        sub = df[df["GROUP"].astype(str).isin(task.groups)]
        n_pt = sub["ID_PT"].nunique()
        counts = sub.groupby("ID_PT")["GROUP"].first().value_counts().to_dict()
        print(f"  {task_id}: pacientes={n_pt} | {counts}")

In [ ]:
# Execução: task × modalidade × ComBat × seleção × modelo × repetições
df_ablation = run_full_ablation_suite(
    base_dir=BASE_DIR,
    roi=ROI,
    tasks=RUN_TASKS,
    modalities=RUN_MODALITIES,
    models=MODELS,
    selection_modes=RUN_SELECTION_MODES,
    with_combat_flags=WITH_COMBAT_FLAGS,
    results_dir=RESULTS_DIR,
    seed=SEED,
    r_repeats=R_REPEATS,
    verbose=VERBOSE,
    combat_quiet=COMBAT_QUIET,
)
df_ablation = prepare_ablation_df(df_ablation)
summary = summary_with_pooled(df_ablation)
print(f"\nResultados: {RESULTS_PATH} ({len(df_ablation)} linhas)")
display(summary[["task", "modality", "model_key", "with_combat", "auc_mean", "auc_std", "auc_pooled"]].head())
df_ablation.head()

In [ ]:
# Resumo com AUC pooled (recomputa se carregou CSV sem rodar execução)
if "df_ablation" not in globals():
    df_ablation = prepare_ablation_df(pd.read_csv(RESULTS_PATH))
summary = summary_with_pooled(df_ablation)
summary.to_csv(SUMMARY_PATH, index=False)
print(f"Resumo salvo em: {SUMMARY_PATH} ({len(summary)} configs)\n")

display_cols = [
    "task", "modality", "with_combat", "model_key",
    "n_outer_evals", "n_repeats", "n_features_mean",
    "auc_mean", "auc_std", "auc_pooled",
    "bal_acc_mean", "sens_pos_mean", "spec_neg_mean",
]
if "selection_mode" in summary.columns:
    display_cols = ["selection_mode"] + display_cols
display(summary[display_cols].head(20))

In [ ]:
# Melhor configuração por task × modalidade (maior AUC pooled)
best_group = ["task", "modality", "with_combat"]
if "selection_mode" in summary.columns:
    best_group = ["selection_mode"] + best_group

best_per_modality = (
    summary
    .sort_values("auc_pooled", ascending=False)
    .groupby(best_group, as_index=False)
    .first()
)
print("Melhor modelo por task × modalidade (AUC pooled):\n")
for _, row in best_per_modality.iterrows():
    mode_prefix = f"[{row['selection_mode']}] " if "selection_mode" in row.index else ""
    combat_tag = "combat" if row.get("with_combat") else "nocombat"
    print(
        f"  {mode_prefix}{row['task']:10s} {row['modality']:8s} ({combat_tag}) "
        f"→ {row['model_key'].upper():4s} | "
        f"AUC pooled={row['auc_pooled']:.3f} | "
        f"AUC mean={row['auc_mean']:.3f}±{row['auc_std']:.3f} | "
        f"feat≈{row['n_features_mean']:.0f}"
    )

In [ ]:
# Comparação visual: AUC por task, modalidade e ComBat
mod_order = ["vol", "texture", "disp", "all"]

for task_id in df_ablation["task"].unique():
    sub_task = df_ablation[df_ablation["task"] == task_id]
    for with_combat in sorted(sub_task["with_combat"].unique()):
        sub = sub_task[sub_task["with_combat"] == with_combat]
        pivot_auc = sub.pivot_table(
            index="model_key", columns="modality", values="auc", aggfunc="mean"
        ).reindex(columns=mod_order)

        fig, ax = plt.subplots(figsize=(8, 4))
        pivot_auc.plot(kind="bar", ax=ax, rot=0)
        combat_tag = "ComBat" if with_combat else "sem ComBat"
        ax.set_title(f"{task_id} — {combat_tag} — AUC médio (5 folds)")
        ax.set_ylabel("AUC")
        ax.legend(title="Modalidade")
        ax.axhline(0.5, color="gray", ls="--", lw=0.8)
        plt.tight_layout()
        out = RESULTS_DIR / f"ablation_auc_{task_id}_{'combat' if with_combat else 'nocombat'}.png"
        # plt.savefig(out, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Figura: {out}")

In [ ]:
# Estabilidade: frequência de cada atributo (0–100%) por configuração isolada
PLOT_ALL_CONFIGS = False  # True = todas; False = só filtros abaixo
PLOT_TASK = "smci_pmci"
PLOT_MODEL = "svm"
PLOT_COMBAT = True
PLOT_SELECTION = "mrmr"

if PLOT_ALL_CONFIGS:
    plot_specs = [
        (str(t), str(m), str(mk), bool(wc))
        for t in df_ablation["task"].astype(str).unique()
        for m in RUN_MODALITIES
        for mk in MODELS
        for wc in WITH_COMBAT_FLAGS
    ]
else:
    plot_specs = [(PLOT_TASK, MODALITY, PLOT_MODEL, PLOT_COMBAT)]

for task_id, mod_id, model_key, with_combat in plot_specs:
    try:
        df_cfg = filter_ablation_config(
            df_ablation,
            task=task_id,
            modality=mod_id,
            model_key=model_key,
            with_combat=with_combat,
            selection_mode=PLOT_SELECTION,
        )
    except ValueError as exc:
        print(f"Pulando {task_id}/{mod_id}/{model_key}/combat={with_combat}: {exc}")
        continue

    freq = feature_freq_table(df_cfg, top_n=None)
    tag = f"{task_id}_{mod_id}_{model_key}_{'combat' if with_combat else 'nocombat'}"
    # freq_path = RESULTS_DIR / f"feature_freq_{tag}.csv"
    # freq.to_csv(freq_path, index=False)

    fig = plot_feature_stability(df_cfg, out_path=RESULTS_DIR / f"feature_stability_{tag}.png")
    plt.show()
    # print(f"Salvo: {freq_path} ({len(freq)} atributos, n_runs={freq['n_runs'].iloc[0]})")
    plt.close(fig)